# 12. Tamaño de entrada en YOLOv8n

La única diferencia de entrenamiento es `imgsz`: 256 frente a la referencia de
640 px en `yolov8n/coco_completo`. Se mantienen COCO, freeze=0, batch 16, semilla
42, augmentation, optimizador, LR, épocas y paciencia del notebook 08.
**El 08 no se modifica** y la referencia no se vuelve a entrenar.

El control se guarda en `yolov8n/coco_completo_256`, con pesos, métricas y
`cajas_val.csv`. Queda fuera de la comparación final de arquitecturas (solo 640 px).
Las corridas archivadas del 27/10 no se reutilizan automáticamente: antes habría
que verificar TODO config.json y documentar la coincidencia en el issue.

Ejecutar en una sesión nueva de Colab con GPU T4. Para leer los resultados y ver
los análisis sin GPU, poner `ENTRENAR=False` y `MEDIR_INFERENCIA=False`.
Este notebook tiene permiso de entrenamiento al montar Drive; su análisis no
escribe archivos en Drive. Para una revisión estrictamente de solo lectura usar 13.

**Val (162) elige el checkpoint y también reporta: las métricas son optimistas.**
Jamaica (23) es un subconjunto de val. N06/N16 mantienen referencia provisional.
Los intervalos no corrigen selección de checkpoints ni correlación espacial.

In [ ]:
ENTRENAR = True
MEDIR_INFERENCIA = False  # medición opcional de ambos modelos en la MISMA sesión T4
# 1. Opciones: comenzar con ambas variantes para cubrir la comparación de la consigna.
REPETIR = False  # True archiva la corrida anterior y vuelve a entrenar.
MODOS = {"coco_completo_256": 0}
SEMILLA = 42
LADO = 256  # Único cambio respecto de la receta de 640 px.
BATCH = 16
EPOCAS = 50
PACIENCIA = 10
UMBRAL = 0.25
CONF_MIN = 0.001  # Bajo para calcular PR/mAP; distinto del umbral de clasificación.
IOU_NMS = 0.7
MAX_DETECCIONES = 300
assert 0 < CONF_MIN < UMBRAL <= 1

In [ ]:
# 2. Código y dependencias: entorno PyTorch separado de los notebooks TensorFlow.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Abrir este notebook en Google Colab con GPU T4.")

repo = Path("/content/PDI-TP")
if repo.is_dir():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
os.chdir(repo)
sys.path.insert(0, str(repo))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(repo / "requirements-detection.txt")], check=True)

import gc
import json
import platform
import shutil
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
from pdi_tp import entorno, resultados, deteccion, clasificacion as clf

if version("ultralytics") != "8.3.221":
    raise RuntimeError("Reiniciar la sesión tras instalar dependencias y volver a ejecutar.")
if ENTRENAR and not torch.cuda.is_available():
    raise RuntimeError("Activar GPU T4 en Entorno de ejecución > Cambiar tipo de entorno.")
rutas = entorno.preparar()
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU; solo análisis")
from pdi_tp import comparacion, estudios, errores

## Datos y receta

Se usa el mismo split, la misma auditoría y la misma copia derivada local que el 08.
El análisis sin entrenamiento lee las anotaciones sin generar el derivado.
Antes de entrenar se comprueba que la referencia completa tenga la receta esperada.

In [ ]:
DERIVADO = repo / "data" / "yolo_paneles_v1"
if ENTRENAR:
    tabla, cajas = deteccion.preparar_dataset(rutas, DERIVADO)
else:
    tabla, cajas = clf.preparar_tabla(rutas)
HUELLA = clf.huella_tabla(tabla)
val = tabla.query("split == 'val'").reset_index(drop=True)
archivos_val = val.archivo.to_list()
fuentes_val = [str(DERIVADO / "images" / "val" / n) for n in archivos_val]
display(tabla.groupby(["split", "fuente", "y_original"]).size().rename("imagenes").to_frame())
print("Cajas incluidas:", int(cajas.incluida.sum()), "Excluidas:", int((~cajas.incluida).sum()))
print("Datos:", DERIVADO / "datos.yaml")

RECETA = dict(imgsz=LADO, batch=BATCH, epochs=EPOCAS, patience=PACIENCIA,
    optimizer="AdamW", lr0=0.001, lrf=0.01, momentum=0.9, weight_decay=0.0005,
    warmup_epochs=3.0, seed=SEMILLA, deterministic=True, workers=2,
    amp=True, cache=False, rect=False, multi_scale=False, fraction=1.0,
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, degrees=90.0, translate=0.1, scale=0.2,
    shear=0.0, perspective=0.0, flipud=0.5, fliplr=0.5,
    mosaic=0.0, mixup=0.0, copy_paste=0.0, close_mosaic=0,
    box=7.5, cls=0.5, dfl=1.5,
    conf=CONF_MIN, iou=IOU_NMS, max_det=MAX_DETECCIONES,
    save=True, save_period=-1, plots=True, val=True, device=0)
ENTORNO = dict(python=platform.python_version(), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    paquetes={p: version(p) for p in ("ultralytics", "torch", "torchvision", "numpy", "pandas")})

In [ ]:
REFERENCIA = "yolov8n/coco_completo"
CONTROL = "yolov8n/coco_completo_256"


def validar_corrida_control(corrida, lado):
    carpeta = rutas.resultados_equipo / corrida
    params = json.loads((carpeta / "config.json").read_text())["params"]
    esperados = dict(arquitectura="yolov8n", inicializacion="COCO yolov8n.pt", freeze=0,
                     huella_manifiesto=HUELLA, umbral_clasificacion=UMBRAL)
    estudios.validar_control(esperados, {c: params[c] for c in esperados if c in params}, [])
    estudios.validar_control({**RECETA, "imgsz": lado}, params.get("receta", {}), [])
    return params


carpeta_ref = rutas.resultados_equipo / REFERENCIA
if resultados.corrida_completa(carpeta_ref):
    validar_corrida_control(REFERENCIA, 640)
elif ENTRENAR:
    raise RuntimeError("Falta la referencia completa a 640 px. Ejecutar primero el notebook 08.")
else:
    print("Falta la referencia de 640 px; no se calcularán diferencias hasta tener ambas corridas.")
if resultados.corrida_completa(rutas.resultados_equipo / CONTROL) and not REPETIR:
    validar_corrida_control(CONTROL, 256)

## Entrenar, evaluar y guardar

Cada variante completa se saltea. Una corrida incompleta retoma `last.pt` si conserva
optimizador; si el entrenamiento terminó, reutiliza `best.pt` y completa la evaluación.
Cambiar la receta exige `REPETIR = True`; se conservan las corridas anteriores.

`freeze=10` congela los módulos 0–9 del backbone de YOLOv8n; `freeze=0` permite ajustar
todos los módulos entrenables (Ultralytics mantiene fija la proyección DFL).
El mejor checkpoint se selecciona por el criterio nativo de detección de esta versión
de Ultralytics. La receta efectiva completa queda en `entrenamiento/args.yaml`.

La evaluación usa confianza mínima 0.001 para conservar cajas de baja confianza al
construir PR/mAP. La clasificación aplica después el umbral 0.25 al máximo por imagen.
Se guardan todas las cajas tras NMS en `cajas_val.csv`, incluso las inferiores al umbral.

Solo se ejecuta el modo `coco_completo_256`, desde COCO; no se parte del modelo de 640.

In [ ]:
def evaluar_detector(modelo, carpeta, yaml_datos, nombre):
    evaluacion = modelo.val(data=str(yaml_datos), split="val", imgsz=LADO, batch=BATCH,
        device=0, workers=2, conf=CONF_MIN, iou=IOU_NMS, max_det=MAX_DETECCIONES,
        augment=False, half=False, plots=True, project=str(carpeta), name=nombre, exist_ok=True)
    resumen = dict(map50=float(evaluacion.box.map50), map50_95=float(evaluacion.box.map),
                   precision=float(evaluacion.box.mp), recall=float(evaluacion.box.mr))
    if not np.isfinite(list(resumen.values())).all():
        raise RuntimeError("La evaluación de detección contiene métricas no finitas.")
    return resumen


if ENTRENAR:
    for modo, congelar in MODOS.items():
        carpeta = resultados.ruta_corrida(rutas, "yolov8n", modo)
        if resultados.corrida_completa(carpeta) and not REPETIR:
            print("Corrida completa, se reutiliza:", carpeta)
            continue
        carpeta = resultados.carpeta_corrida(rutas, "yolov8n", modo, reiniciar=REPETIR)
        params = dict(arquitectura="yolov8n", inicializacion="COCO yolov8n.pt", freeze=congelar,
            receta=RECETA, huella_manifiesto=HUELLA, entorno=ENTORNO,
            criterio_datos="v1: clase panel; excluir solo Z01/linea 1; mantener cajas pequenas",
            evaluacion="val selecciona checkpoint y reporta; N06/N16 provisionales",
            puntaje_clasificacion="maxima confianza de cajas tras NMS; 0 sin detecciones; no calibrado",
            umbral_clasificacion=UMBRAL)
        # Evita mezclar entrenamiento reanudado con otra receta o dataset.
        solicitud = carpeta / "solicitud.json"
        if solicitud.exists() and json.loads(solicitud.read_text()) != params:
            raise RuntimeError("Cambió la configuración. Usar REPETIR=True para archivar la anterior.")
        solicitud.write_text(json.dumps(params, indent=2), encoding="utf-8")
        shutil.copy2(DERIVADO / "manifiesto.csv", carpeta / "manifiesto.csv")
        shutil.copy2(DERIVADO / "cajas_derivadas.csv", carpeta / "cajas_derivadas.csv")
        entrenamiento = carpeta / "entrenamiento"
        ultimo = entrenamiento / "weights" / "last.pt"
        mejor = entrenamiento / "weights" / "best.pt"
        if ultimo.exists():
            # Solo checkpoints propios creados por este notebook.
            estado = torch.load(ultimo, map_location="cpu", weights_only=False)
            epoca = int(estado["epoch"])
            del estado
            if epoca >= 0:
                if not mejor.exists():
                    raise RuntimeError("Falta best.pt: conservar archivos y revisar antes de reanudar.")
                print("Reanudando desde last.pt:", modo)
                modelo = YOLO(str(ultimo))
                modelo.train(resume=True, data=str(DERIVADO / "datos.yaml"), device=0)
            else:
                print("Entrenamiento finalizado; se recupera best.pt para evaluar:", modo)
        elif mejor.exists() or (entrenamiento / "results.csv").exists():
            raise RuntimeError("Hay historial o best.pt pero falta last.pt; no se sobrescribe la corrida.")
        else:
            modelo = YOLO("yolov8n.pt")
            modelo.train(data=str(DERIVADO / "datos.yaml"), project=str(carpeta),
                         name="entrenamiento", exist_ok=True, freeze=congelar, **RECETA)
        if not mejor.is_file():
            raise RuntimeError("No se generó best.pt; revisar el entrenamiento.")
        shutil.copy2(mejor, carpeta / "modelo.pt")
        modelo = YOLO(str(carpeta / "modelo.pt"))
        metricas_det = {
            "val": evaluar_detector(modelo, carpeta, DERIVADO / "datos.yaml", "deteccion_val"),
            "jamaica": evaluar_detector(modelo, carpeta, DERIVADO / "datos_jamaica.yaml", "deteccion_jamaica"),
        }
        salidas = modelo.predict(source=fuentes_val, stream=True, imgsz=LADO, batch=BATCH,
            device=0, conf=CONF_MIN, iou=IOU_NMS, max_det=MAX_DETECCIONES,
            augment=False, half=False, save=False, verbose=False)
        puntajes, filas = deteccion.resumir_predicciones(salidas, archivos_val)
        pd.DataFrame(filas, columns=["archivo", "clase", "confianza", "xc", "yc", "w", "h"]).to_csv(
            carpeta / "cajas_val.csv", index=False)
        clf.guardar_json(carpeta / "metricas_deteccion.json", metricas_det)
        mascara = ~val.pendiente_revision.to_numpy()
        extra = dict(deteccion=metricas_det,
            sin_pendientes=clf.evaluar(val.loc[mascara, "y_original"].to_numpy(dtype=int),
                                       puntajes[mascara], UMBRAL))
        resultados.guardar_corrida(rutas, carpeta, archivos=archivos_val, probabilidad=puntajes,
                                  params=params, umbral=UMBRAL, metricas_extra=extra)
        display(pd.DataFrame(metricas_det).T)
        del modelo
        gc.collect()
        torch.cuda.empty_cache()

## Detección y clasificación sobre val y Jamaica

Se leen mAP50, mAP50-95, precision y recall de detección ya guardados por Ultralytics.
Su precision/recall corresponden al criterio nativo de la curva; el recall por tamaño
se calcula aparte con confianza fija 0.25 e IoU ≥ 0.5. No son intercambiables.

Las diferencias de clasificación son **256 menos 640**, con IC95% pareado por imagen.
Val también seleccionó los checkpoints. Las tablas no eligen un ganador de antemano.

In [ ]:
corridas = []
for corrida, lado in ((REFERENCIA, 640), (CONTROL, 256)):
    if resultados.corrida_completa(rutas.resultados_equipo / corrida):
        validar_corrida_control(corrida, lado)
        corridas.append(corrida)
    else:
        print("Pendiente de entrenar/completar:", corrida)
if corridas:
    display(comparacion.tabla_deteccion(rutas.resultados_equipo, corridas).round(4))
    display(estudios.tabla_metricas(rutas.resultados_equipo, corridas).round(4))
if len(corridas) == 2:
    predicciones = resultados.cargar_predicciones(rutas)
    for jamaica in (False, True):
        print("Jamaica (23)" if jamaica else "Val (162)", "· 256 − 640 · IC95%")
        display(pd.concat([comparacion.diferencias(predicciones, corridas, REFERENCIA, metrica=m,
            solo_jamaica=jamaica, n=2000, semilla=42) for m in estudios.METRICAS], ignore_index=True).round(4))

## Recall por tamaño de caja en píxeles originales

Lado = raíz del área de la caja en la imagen original, no en la entrada redimensionada.
Bins: <16, [16,32), [32,64), ≥64 px. Se emparejan todas las cajas de una imagen
por confianza descendente antes de separar tamaños; cada predicción acierta como
máximo una referencia. Confianza ≥0.25 e IoU ≥0.5 en ambas corridas.

El bootstrap remuestrea imágenes completas (incluye las que no tienen cajas), con
los mismos índices para ambos modelos. Un bin sin referencias es indefinido, no cero;
se informa cuántos remuestreos tuvieron cajas. Si el IC de la diferencia incluye
cero, no hay evidencia de mejora en ese grupo. No se calcula un IC de mAP con este
recall diagnóstico; la tabla de mAP por sí sola no acredita significación estadística.

In [ ]:
cajas_control = {}
if len(corridas) == 2:
    inventario, pred_revision, exportaciones = errores.cargar_revision(
        rutas, tabla, incluir_detectores_anteriores=True)
    estado = inventario.set_index("corrida")
    for corrida in corridas:
        if corrida not in estado.index or not estado.loc[corrida, "disponible"]:
            motivo = estado.loc[corrida, "motivo"] if corrida in estado.index else "Sin exportación"
            raise ValueError(f"{corrida}: {motivo}. No equivale a cero detecciones.")
        cajas_control[corrida] = exportaciones[corrida]
    aciertos = estudios.aciertos_por_caja(tabla, cajas, cajas_control, confianza=UMBRAL, iou_min=0.5)
    display(estudios.recall_por_tamano(tabla, aciertos, corridas, REFERENCIA, n=2000).round(4))

## Costo medido

Tiempo por época: diferencias del reloj acumulado en `entrenamiento/results.csv`.
Se informa GPU y entorno de cada corrida: no atribuir un cambio a resolución si
cambió el hardware o el entorno. Si el reloj se reinició al reanudar, se informa
que falta una medición comparable; no se estima.

Inferencia: activar `MEDIR_INFERENCIA=True` en T4. Ambos modelos se miden en la misma
sesión, batch 1, FP32, mismos umbrales/NMS, todas las imágenes de val y tres pasadas
tras calentamiento. Se sincroniza CUDA; se excluyen lectura de Drive, carga de pesos
y calentamiento. Se incluyen preprocesado e inferencia/NMS. La medición queda en
memoria: conservar su salida en una copia personal del notebook, sin subirla a git.

In [ ]:
filas_tiempo = []
for corrida in corridas:
    carpeta = rutas.resultados_equipo / corrida
    params = json.loads((carpeta / "config.json").read_text())["params"]
    historial = carpeta / "entrenamiento" / "results.csv"
    if not historial.is_file():
        print("Sin tiempos de entrenamiento:", corrida)
        continue
    historia = pd.read_csv(historial)
    historia.columns = historia.columns.str.strip()
    try:
        segundos = estudios.tiempos_por_epoca(historia)
    except (KeyError, ValueError) as error:
        print(corrida, "· tiempo no comparable:", error)
        continue
    filas_tiempo.append(dict(corrida=corrida, epocas=len(segundos),
        segundos_por_epoca_media=segundos.mean(), segundos_por_epoca_mediana=np.median(segundos),
        gpu=params.get("entorno", {}).get("gpu"), paquetes=params.get("entorno", {}).get("paquetes")))
if filas_tiempo:
    display(pd.DataFrame(filas_tiempo))

if MEDIR_INFERENCIA:
    import time
    from PIL import Image
    if len(corridas) != 2 or not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
        raise RuntimeError("La medición requiere ambas corridas completas y una GPU T4.")
    imagenes = []
    for archivo in archivos_val:
        with Image.open(rutas.imagenes / archivo) as imagen:
            imagenes.append(imagen.convert("RGB"))
    tiempos_inferencia = []
    for corrida, lado in ((REFERENCIA, 640), (CONTROL, 256)):
        modelo = YOLO(str(rutas.resultados_equipo / corrida / "modelo.pt"))
        opciones = dict(imgsz=lado, batch=1, device=0, conf=CONF_MIN, iou=IOU_NMS,
                        max_det=MAX_DETECCIONES, augment=False, half=False, save=False, verbose=False)
        for _ in range(3):
            modelo.predict(source=imagenes[:1], **opciones)
        for pasada in range(3):
            torch.cuda.synchronize()
            inicio = time.perf_counter()
            for imagen in imagenes:
                modelo.predict(source=imagen, **opciones)
            torch.cuda.synchronize()
            ms = 1000 * (time.perf_counter() - inicio) / len(imagenes)
            tiempos_inferencia.append(dict(corrida=corrida, pasada=pasada + 1, ms_por_imagen=ms,
                gpu=torch.cuda.get_device_name(0), batch=1, precision="FP32", imagenes=len(imagenes),
                paquetes=ENTORNO["paquetes"]))
        del modelo
        gc.collect()
        torch.cuda.empty_cache()
    display(pd.DataFrame(tiempos_inferencia))

## Hasta cuatro ejemplos reales de cajas pequeñas

Se eligen imágenes con cajas de lado <32 px cuyo acierto difiere entre 256 y 640,
incluyendo ambos sentidos. Se muestra la primera caja discrepante por imagen y
el mismo zoom para ambas corridas; no son una muestra representativa.
N06/N16 se excluyen de los ejemplos por su referencia provisional, pero se conservan
en las métricas del split. Si no hay casos, se dice explícitamente.
También se pueden explorar en `14_galeria_errores` habilitando otra resolución.

In [ ]:
if cajas_control:
    pendientes = set(tabla.loc[tabla.pendiente_revision, "archivo"])
    candidatos = aciertos.loc[(aciertos.lado < 32) & (aciertos[CONTROL] != aciertos[REFERENCIA])
                             & ~aciertos.archivo.isin(pendientes)].copy()
    candidatos["sentido"] = np.where(candidatos[CONTROL], "acierta 256", "acierta 640")
    ejemplos = candidatos.sort_values(["sentido", "lado"]).drop_duplicates("archivo")
    # Alternar sentidos cuando existen; no buscar solamente ejemplos favorables a 640.
    grupos = [g for _, g in ejemplos.groupby("sentido", sort=True)]
    orden = [g.iloc[i] for i in range(4) for g in grupos if i < len(g)][:4]
    if not orden:
        print("No hay cajas pequeñas con acierto diferente entre las corridas a estos umbrales.")
    for caso in orden:
        reales = cajas.loc[(cajas.archivo == caso.archivo) & cajas.incluida, errores.COORDENADAS].to_numpy(float)
        diagnosticos = {c: errores.diagnosticar_cajas(reales, p.loc[p.archivo == caso.archivo],
                        confianza=UMBRAL, iou_min=0.5) for c, p in cajas_control.items()}
        caja = reales[int(caso.caja)]
        zoom = np.r_[caja[:2] - caja[2:]/2, caja[:2] + caja[2:]/2]
        figura = errores.mostrar_imagen(rutas, caso.archivo, reales, diagnosticos, zoom=zoom,
            mostrar_aciertos=True, titulo=f"{caso.sentido} · lado {caso.lado:.1f} px · selección ilustrativa")
        plt.show()
        plt.close(figura)

## Para completar el informe

Registrar corridas, fecha, commit y entorno realmente usados. No hay resultados
precargados ni una conclusión sobre cuál resolución gana. Si el recall de cajas
chicas no muestra una diferencia medible, decirlo y considerar el costo. El mAP
puntual no tiene aquí un intervalo de incertidumbre; no afirmar significación
estadística a partir de dos valores. Val seleccionó el checkpoint y también reportó.
Mantener pendiente el issue #36 hasta ejecutar el control y revisar sus resultados.